In [1]:
import os
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.engine.modes.drill_mode import SoloDrillMode
from src.rl.env_wrapper import HaxballGymEnv
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import ProximalStrikerReset
from src.rl.reward_shapers import Stage1Reward
from src.rl.trainer import train_ppo_vectorized

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def make_env():
    match_cfg = MatchConfig(
        mode=SoloDrillMode(),
        roster=[PlayerSlot(team="red", stats=PlayerStats(name="Agent"))],
    )
    return HaxballGymEnv(
        match_config=match_cfg,
        reward_shaper=Stage1Reward(),
        reset_strategy=ProximalStrikerReset(),
        max_steps=300,
    )


# 1. Parallel environments for training exploration
num_envs = 16
train_envs = gym.vector.AsyncVectorEnv(
    [make_env for _ in range(num_envs)]
)

# 2. Dedicated single environment for fixed-seed validation
eval_env = make_env()

# 3. Model setup
model = ActorCritic(obs_dim=68).to(device)



In [2]:
# 4. Train with 50-episode deterministic evaluation every 50,000 steps
train_ppo_vectorized(
    envs=train_envs,
    eval_env=eval_env,
    model=model,
    device=device,
    total_timesteps=5_000_000,
    num_envs=num_envs,
    eval_freq=100_000,
    eval_episodes=100,
    save_dir="models/stage1",
    lr_initial=3e-4,
    lr_final=1e-5,
)

🚀 Training with Deterministic Validation (100 eps every 100000 steps)...

📊 [EVALUATION @ Step  102400] Goal Rate:  12.0% (12/100) | Touch Rate:  66.0% | Avg Steps: 273.7 | Mean Reward: -33.72
   ⭐ New verified best model saved: models/stage1/best_stage1.pt (12.0% goals)


📊 [EVALUATION @ Step  200704] Goal Rate:  23.0% (23/100) | Touch Rate:  56.0% | Avg Steps: 246.4 | Mean Reward: -16.68
   ⭐ New verified best model saved: models/stage1/best_stage1.pt (23.0% goals)


📊 [EVALUATION @ Step  303104] Goal Rate:  37.0% (37/100) | Touch Rate:  83.0% | Avg Steps: 219.9 | Mean Reward:  13.27
   ⭐ New verified best model saved: models/stage1/best_stage1.pt (37.0% goals)


📊 [EVALUATION @ Step  401408] Goal Rate:  42.0% (42/100) | Touch Rate:  88.0% | Avg Steps: 208.2 | Mean Reward:  25.50
   ⭐ New verified best model saved: models/stage1/best_stage1.pt (42.0% goals)


📊 [EVALUATION @ Step  503808] Goal Rate:  49.0% (49/100) | Touch Rate:  92.0% | Avg Steps: 203.6 | Mean Reward:  36.34
   ⭐ Ne

In [6]:
from IPython.display import IFrame
from src.rl.evaluator import evaluate_and_generate_html

# 1. Instantiate test environment directly (single call)
test_env = make_env()

# 2. Evaluate best checkpoint across 5 episodes
replay_file = evaluate_and_generate_html(
    env=test_env,
    model_or_path="models/stage1/best_stage1.pt",
    device=device,
    output_dir="renders/stage1",
    filename="stage1_best_eval.html",
    num_episodes=10,
    max_steps=300,
)

test_env.close()

# 3. Display interactive player directly inside the notebook
IFrame(src=replay_file, width=960, height=750)

🎬 Interactive replay saved to: renders/stage1/stage1_best_eval.html
